<a href="https://colab.research.google.com/github/nasaharvest/crop-cycle-detection/blob/main/examples/01_byod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crop Cycle Detection — Bring Your Own Data

This notebook demonstrates the **bring-your-own-data** workflow:
provide a DataFrame with `date` and `NDVI` columns and get back
per-season milestones — season start/end, peak NDVI date, and green-up date.

**No GEE account or cloud credentials required.**

In [ ]:
# ── Colab setup (safe to run locally too — the `if` guards it) ──────────────
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone https://github.com/nasaharvest/crop-cycle-detection.git /content/crop-cycle-detection
    %cd /content/crop-cycle-detection/examples
    !pip install -q -r ../requirements.txt

# Make src/ importable from examples/
sys.path.insert(0, '..')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.crop_cycle import crop_cycle_model

## 1. Load the sample NDVI time series

The sample file contains ~2 years of bi-weekly Sentinel-2 / Landsat observations
for a single agricultural field.  Replace it with your own CSV — any file with
`date` and `NDVI` columns works.

In [ ]:
df = pd.read_csv("../sample_data/example_ndvi.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Observations: {len(df)}")
print(f"Date range:   {df['date'].min().date()} → {df['date'].max().date()}")
df.head()

## 2. Run the crop cycle model

In [ ]:
report, ndvi_processed = crop_cycle_model(df)

print(f"\nDetected {len(report)} season(s)\n")
display_cols = [
    "season_number", "approx_season_start", "approx_season_end",
    "approx_season_length", "greenup_date", "ndvi_peak_date", "ndvi_peak_value",
]
report[display_cols]

## 3. Visualise the results

The plot shows:
- Raw NDVI observations (dots)
- Smoothed / interpolated NDVI (line)
- Detected season boundaries and peak NDVI dates (vertical lines)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Raw observations
ax.scatter(df["date"], df["NDVI"], s=18, color="steelblue", alpha=0.6,
           zorder=3, label="Observed NDVI")

# Smoothed series
ax.plot(ndvi_processed["date"], ndvi_processed["NDVI_interpolated"],
        color="steelblue", linewidth=1.5, label="Smoothed NDVI")

# Season milestones
colors = plt.cm.tab10.colors
for i, row in report.iterrows():
    c = colors[i % len(colors)]
    label_prefix = f"Season {int(row['season_number'])}"

    if pd.notna(row["approx_season_start"]):
        ax.axvline(row["approx_season_start"], color=c, linestyle="--",
                   linewidth=1.2, label=f"{label_prefix} start")
    if pd.notna(row["approx_season_end"]):
        ax.axvline(row["approx_season_end"], color=c, linestyle=":",
                   linewidth=1.2, label=f"{label_prefix} end")
    if pd.notna(row["ndvi_peak_date"]):
        ax.axvline(row["ndvi_peak_date"], color=c, linestyle="-",
                   linewidth=1.8, alpha=0.7, label=f"{label_prefix} peak")
    if pd.notna(row.get("greenup_date")):
        ax.axvline(pd.Timestamp(row["greenup_date"]), color=c,
                   linestyle="-.", linewidth=1.0, alpha=0.5,
                   label=f"{label_prefix} green-up")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax.set_xlabel("Date")
ax.set_ylabel("NDVI")
ax.set_title("Crop Cycle Detection")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 4. Inspect all output columns

In [ ]:
report

## 5. Using your own data

Replace the CSV path above with your own file.  The only required columns are `date` and `NDVI`:

```python
df = pd.read_csv("your_field.csv", parse_dates=["date"])
report, ndvi_processed = crop_cycle_model(df)
```

To process multiple fields, call `crop_cycle_model` in a loop:

```python
results = []
for field_name, group in df_all.groupby("field_id"):
    group = group.copy()
    group["poly_name"] = field_name
    r, _ = crop_cycle_model(group)
    results.append(r)

all_seasons = pd.concat(results, ignore_index=True)
```

See the [README](../README.md) for a full list of tunable parameters.